# Melt function in Pandas
Melt function just like ```pivot``` function is used to reshape the dataFrame

In [11]:
import pandas as pd
import numpy as np
import itertools
from matplotlib import pyplot as plt
from typing import List, Optional

weather_df = pd.read_csv("/home/aditya/github/Deep-learning-prerequisite/pandas/dataset/weather_data_melt_func_demo.csv")
weather_df

,day,chicago,chennai,berlin,tokyo,sydney
0,Monday,32,75,41,55,65
1,Tuesday,30,77,43,58,68
2,Wednesday,28,75,45,53,64
3,Thursday,22,82,38,50,63
4,Friday,30,83,30,56,66
5,Saturday,20,81,45,54,67
6,Sunday,25,77,47,52,65


In [12]:
melted_df = pd.melt(weather_df,id_vars=["day"])
melted_df

,day,variable,value
0,Monday,chicago,32
1,Tuesday,chicago,30
2,Wednesday,chicago,28
3,Thursday,chicago,22
4,Friday,chicago,30
5,Saturday,chicago,20
6,Sunday,chicago,25
7,Monday,chennai,75
8,Tuesday,chennai,77
9,Wednesday,chennai,75


### Huh? ... Interesting what do we have here 
### Let's use pivot on this ```melted_df``` and see if my gut feeling right  

In [13]:
pivoted_df = melted_df.pivot(index="day",columns="variable",values="value")
pivoted_df

variable,berlin,chennai,chicago,sydney,tokyo
day,,,,,
Friday,30,83,30,66,56
Monday,41,75,32,65,55
Saturday,45,81,20,67,54
Sunday,47,77,25,65,52
Thursday,38,82,22,63,50
Tuesday,43,77,30,68,58
Wednesday,45,75,28,64,53


### I knew it! ```pd.melt()``` function is nothing but opposite to ```df.pivot()``` function. 
### Theory:
- ```pd.melt()``` this function reshapes a DataFrame by converting columns into rows.
- Wide format -> many columns for variable
- Long format -> fewer columns, but more rows (tidy)
#### What is ```pd.melt()```:
- Your columns represent variable names (like years, months, metrics) instead of values.
- You want to prepare data for plotting (most plotting libraries prefer long format).
- You want to normalize messy spreadsheets into a standard structure.
#### When not to use ```pd.melt()```:
- Your data is already in long/tidy format.
- You want a pivoted summary (example, aggregating sales per product).
- You need aggregations/grouping -> use ```pivot_table``` instead
### Analysis
```pd.melt()``` function is also used to rehape the dataFrame just like the ```pivot``` function.
- First argument taken by this function is the dataFrame.
- Second argument taken by this function is ```id_vars[]```. The id variables is basically the thing that you want in your x axis in other words you pass in the columns that you want to keep intact after the rehaping of the dataFrame is complete.

#### How to set variable name and value name in melt function

In [14]:
melt_df = pd.melt(weather_df,id_vars=["day"],var_name="city",value_name="temperature")
melt_df

,day,city,temperature
0,Monday,chicago,32
1,Tuesday,chicago,30
2,Wednesday,chicago,28
3,Thursday,chicago,22
4,Friday,chicago,30
5,Saturday,chicago,20
6,Sunday,chicago,25
7,Monday,chennai,75
8,Tuesday,chennai,77
9,Wednesday,chennai,75


## Exmaple 1: Sales by year

### Generate data

In [24]:
def generate_sales_data(products:List[str]=None, 
                        start_year:int=None, 
                        end_year:int=None, 
                        seed:int=None)->pd.DataFrame:
    # check if the arguments passed in this function is None or not
    if not products:
        raise ValueError("products is required!")
    if not start_year:
        raise ValueError("start_year is required!")
    if not end_year:
        raise ValueError("end_year is required!")
    if not seed:
        raise ValueError("seed is required!")
        
    # check the type of each arguments that is passed into this function
    if not isinstance(products,list):
        raise TypeError("products must be of type list")
    if not isinstance(start_year,int):
        raise TypeError("start_year must be of type int")
    if not isinstance(end_year,int):
        raise TypeError("end_year must be of type int")
    if not isinstance(seed,int):
        raise TypeError("seed must be of type int")
    
    np.random.seed(seed)  # for reproducibility
    
    # Years as columns
    years = list(map(str, range(start_year, end_year + 1)))
    
    # Random sales data between 100 and 5000
    data = {year: np.random.randint(100, 5000, size=len(products)) for year in years}
    
    # Build DataFrame
    df = pd.DataFrame(data)
    df.insert(0, "product", products)  # insert product column at start
    
    return df

In [25]:
products = [f"Product_{i}" for i in range(1, 31)] 
start_year=2004
end_year=2025
seed=42
sales_df = generate_sales_data(products,start_year,end_year,seed)
sales_df

,product,2004,2005,2006,2007,2008,2009,2010,2011,2012,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Product_1,960,1999,3943,3661,1159,3843,3031,723,3877,...,930,1343,2031,3246,2602,1317,1181,1496,1938,2148
1,Product_2,3872,1367,1600,2378,1822,1595,297,1116,2969,...,3775,3218,3846,512,1919,1640,555,3737,2044,4246
2,Product_3,3192,1628,4898,3199,3414,3404,2030,3743,3334,...,1469,2241,2527,828,817,3336,994,3430,109,4880
3,Product_4,566,3302,261,300,3257,3863,3682,2149,3891,...,840,3159,1668,3448,2882,2486,1484,2803,2971,816
4,Product_5,4526,3656,4397,4975,2725,1953,708,3823,1056,...,216,4648,4100,2908,3677,3819,2058,570,897,4644
5,Product_6,3544,3990,2081,3204,2829,4759,3372,4648,4167,...,2938,3059,2838,775,243,1488,3172,1453,1115,4683
6,Product_7,3271,746,1095,4698,1697,1391,1247,3208,260,...,1250,2365,3598,3565,3240,1956,2096,2883,1924,1923
7,Product_8,3019,2988,3442,2554,3160,3681,4382,4190,4593,...,2031,2579,4013,2604,4591,1465,4385,2931,4569,2181
8,Product_9,230,2535,4651,3745,2793,3557,3497,2156,1400,...,1112,4746,1166,3067,3038,599,604,457,3236,3519
9,Product_10,1785,700,3898,904,4991,1736,2611,3270,4087,...,1764,1735,4480,2886,3736,1780,1314,3119,2293,1674


### Apply melt function

In [26]:
melted_sales = pd.melt(sales_df,id_vars=["product"],var_name="year",value_name="sales")
melted_sales

,product,year,sales
0,Product_1,2004,960
1,Product_2,2004,3872
2,Product_3,2004,3192
3,Product_4,2004,566
4,Product_5,2004,4526
...,...,...,...
655,Product_26,2025,2914
656,Product_27,2025,3474
657,Product_28,2025,3378
658,Product_29,2025,1352


### Apply groupby function

In [27]:
grouped_sales_df=melted_sales.groupby("product")
grouped_sales_df

### Plot the data using matplotlib

In [ ]:
# create a list of colors
color_list = ["green","purple","red","magenta","blue","yellow","orange"]
# create a list of line styles
linestyle_list= ["-","--",":","-.",(0,(5,1,5,1)),(0,(5,10))]

# create a cyclic iterator for color_list

# create a cyclic iterator for linestyle_lits

plt.figure(figsize=(12,6))
for product,df in grouped_sales_df:
    plt.plot(df["year"],df["sales"],marker="o",label="product")